# Notebook to convert .csv file to .parquet for groups database

In [ ]:
import pandas as pd



# Konstanter
FILE_PATH = "../data/raw/clean_music_format_year.csv"

df = pd.read_csv(FILE_PATH)

In [5]:
print("\n=== DTYPES (Data types) ===")
print(df.dtypes)



=== DTYPES (Data types) ===
Format                str
Metric                str
Year                int64
Value (Actual)    float64
dtype: object


In [6]:
# ==========================================
# 1. SQL-VÄNLIGA KOLUMNNAMN
# ==========================================
df = df.rename(columns={
    'Format': 'format',
    'Metric': 'metric',
    'Year': 'year',
    'Value (Actual)': 'value'
})

print("=== NYA CLEANSED DTYPES ===")
print(df.dtypes)

# ==========================================
# 2. Spara till processed
# ==========================================
silver_media_path = "../data/processed/historical_media_sales.parquet"
df.to_parquet(silver_media_path, index=False)

print(f"\nFjärde datasetet säkrat i Silver: {silver_media_path}")

=== NYA CLEANSED DTYPES ===
format        str
metric        str
year        int64
value     float64
dtype: object

Fjärde datasetet säkrat i Silver: ../data/processed/historical_media_sales.parquet


In [7]:
import duckdb
# Sökvägen till din databas
DB_PATH = "../data/music_warehouse.duckdb" 

# Koppla upp (read_only=True så vi inte råkar förstöra något)
con = duckdb.connect(DB_PATH, read_only=True)

# Hämta schemat för de tre nya tabellerna
print("=== TABELL 2: HISTORICAL CHARTS ===")
print(con.execute("DESCRIBE silver_historical_charts;").df()[['column_name', 'column_type']])

print("\n=== TABELL 3: TOP 200 HISTORICAL ===")
print(con.execute("DESCRIBE silver_top_200_historical;").df()[['column_name', 'column_type']])

print("\n=== TABELL 4: MEDIA SALES ===")
print(con.execute("DESCRIBE silver_music_format_sales;").df()[['column_name', 'column_type']])

con.close()

=== TABELL 2: HISTORICAL CHARTS ===
     column_name column_type
0           name     VARCHAR
1           rank      BIGINT
2  snapshot_date   TIMESTAMP
3        artists     VARCHAR
4        country     VARCHAR
5          chart     VARCHAR
6          trend     VARCHAR
7        streams      DOUBLE

=== TABELL 3: TOP 200 HISTORICAL ===
                      column_name column_type
0                            rank      BIGINT
1                            name     VARCHAR
2                         artists     VARCHAR
3                   snapshot_date   TIMESTAMP
4                    danceability      DOUBLE
5                          energy      DOUBLE
6                        loudness      DOUBLE
7                     speechiness      DOUBLE
8                    acousticness      DOUBLE
9                instrumentalness      DOUBLE
10                        valence      DOUBLE
11                     artist_ind     VARCHAR
12                    nationality     VARCHAR
13                   